<a href="https://colab.research.google.com/github/AmiriHayes/rockphysicsproject/blob/main/graph_ml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## __GROUP 2 ML Code:__

GOAL - Use machine learning to map representation to permeability \
* https://chatgpt.com/share/67c10964-8b3c-8004-9dbb-f566bc51a04c \
     -  Method 1: Try Node2Vec and Regular Deep Neural Network \
     -  Method 2: Try GCN: Graph Convolutional Networks \
     -  Method 3: Try MPNN: Message-Passing Neural Network \

## __References:__
- Group 2 Report: [PoreSPY Generation Presentation](https://google.com)
- Group 2 Report: [SNOW2 Networks Presentation](https://docs.google.com/presentation/d/1SSHW31IZ6fWqkfT2cLFDcU22-vQ2aC4XyXFaqe7JiCk/edit?usp=sharing)


In [4]:
!pip install node2vec numpy
from node2vec import Node2Vec

import torch
import os
import pickle
import numpy as np
import pandas as pd
import networkx as nx
import tensorflow as tf
import sklearn

np.random.seed(10)

In [20]:
# COLLECT NETWORKS

# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Replace with the actual path to your "networks" folder
networks_folder_path = '/content/drive/My Drive/networks'

X = []  # Use a list to store all arrays

if os.path.exists(networks_folder_path) and os.path.isdir(networks_folder_path):
    for i in range(1000):
        filename = f"network_{i:03d}.pkl"  # Assuming files are named network_0.pkl to network_999.pkl
        file_path = os.path.join(networks_folder_path, filename)
        if os.path.exists(file_path):
            try:
                with open(file_path, 'rb') as f:
                    data = pickle.load(f)
                    X.append(data)
                print(f"Successfully unpickled: {filename}")
            except Exception as e:
                print(f"Error unpickling {filename}: {e}")
        else:
            print(f"Warning: File not found: {filename}")
    print(f"\nTotal number of arrays unpickled: {len(X)}")
else:
    print(f"Error: Folder not found or is not a directory at path: {networks_folder_path}")

ERROR:openpnm:PARDISO solver not installed, run `pip install pypardiso`. Otherwise, simulations will be slow. Apple M chips not supported.


Successfully unpickled: network_000.pkl
Successfully unpickled: network_001.pkl
Successfully unpickled: network_002.pkl
Successfully unpickled: network_003.pkl
Successfully unpickled: network_004.pkl
Successfully unpickled: network_005.pkl
Successfully unpickled: network_006.pkl
Successfully unpickled: network_007.pkl
Successfully unpickled: network_008.pkl
Successfully unpickled: network_009.pkl
Successfully unpickled: network_010.pkl
Successfully unpickled: network_011.pkl
Successfully unpickled: network_012.pkl
Successfully unpickled: network_013.pkl
Successfully unpickled: network_014.pkl
Successfully unpickled: network_015.pkl
Successfully unpickled: network_016.pkl
Successfully unpickled: network_017.pkl
Successfully unpickled: network_018.pkl
Successfully unpickled: network_019.pkl
Successfully unpickled: network_020.pkl
Successfully unpickled: network_021.pkl
Successfully unpickled: network_022.pkl
Successfully unpickled: network_023.pkl
Successfully unpickled: network_024.pkl


In [21]:
# COLLECT PERMEABILITY

permeability_file_path = '/content/drive/MyDrive/snow_perms.txt'
Y = []

try:
    with open(permeability_file_path, 'r') as f:
        for line in f:
            # Remove leading/trailing whitespace and convert to float
            permeability_value = float(line.strip())
            Y.append(permeability_value)
    print(f"Successfully loaded permeability values from: {permeability_file_path}")
    Y = np.array(Y)  # Convert the list to a NumPy array
    print(f"Shape of the permeability array Y: {Y.shape}")
except FileNotFoundError:
    print(f"Error: File not found at path: {permeability_file_path}")
except ValueError:
    print("Error: Could not convert all lines in the file to float. Ensure each line contains a single numeric value.")
except Exception as e:
    print(f"An error occurred: {e}")

Successfully loaded permeability values from: /content/drive/MyDrive/snow_perms.txt
Shape of the permeability array Y: (1000,)


##METHOD ONE: EMBEDDINGS & ORDINARY REGRESSION

In [ ]:
# CONVERT NETWORK DATA TO EMBEDDINGS

!pip install openpnm as op
import openpnm as op

def generate_embeddings(graph, dimensions=128, walk_length=30, num_walks=200, window_size=10, workers=4):
    """Generate node2vec embeddings for a single graph."""
    node2vec = Node2Vec(
        graph,
        dimensions=dimensions,
        walk_length=walk_length,
        num_walks=num_walks,
        workers=workers
    )

    model = node2vec.fit(window=window_size, min_count=1, batch_words=4)
    embedding = np.mean([model.wv[str(node)] for node in graph.nodes], axis=0)
    return embedding

X_final = []
for i in range(len(X)):
    raw_network = op.io.network_from_porespy(X[i]["network"])
    nx_graph = op.io.network_to_networkx(raw_network)
    embedding = generate_embeddings(nx_graph)
    X_final.append(embedding)

ERROR: Could not find a version that satisfies the requirement as (from versions: none)
ERROR: No matching distribution found for as


Computing transition probabilities:   0%|          | 0/327 [00:00<?, ?it/s]

Computing transition probabilities:   0%|          | 0/206 [00:00<?, ?it/s]

Computing transition probabilities:   0%|          | 0/148 [00:00<?, ?it/s]

Computing transition probabilities:   0%|          | 0/112 [00:00<?, ?it/s]

Computing transition probabilities:   0%|          | 0/147 [00:00<?, ?it/s]

Computing transition probabilities:   0%|          | 0/143 [00:00<?, ?it/s]

Computing transition probabilities:   0%|          | 0/212 [00:00<?, ?it/s]

In [ ]:
# LINEAR REGRESSION

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
import seaborn as sns
import matplotlib.pyplot as plt

np.random.seed(42)
n_samples = 100

X_train, X_test, y_train, y_test = train_test_split(X_final, Y, test_size=0.2, random_state=42)

print(f"Check data:\nMax embedding value (train): {np.max(X_train)}")
print(f"Min embedding value (train): {np.min(X_train)}\n")

print(f"Check data:\nMax permeability value (train): {np.max(y_train)}")
print(f"Min permeability value (train): {np.min(y_train)}\n")

lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

y_pred = lr_model.predict(X_test)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(f"R²: {r2:.4f}")
print(f"MAE: {mae:.4f}")

# Residual Plot
plt.figure(figsize=(10, 6))
residuals = y_test - y_pred
plt.scatter(y_pred, residuals, color='green')
plt.axhline(y=0, color='red', linestyle='--')
plt.xlabel('Predicted Permeability')
plt.ylabel('Residuals (Actual - Predicted)')
plt.title('Residual Plot')
plt.grid(True)
plt.show()

# Histogram of Prediction Errors
plt.figure(figsize=(10, 6))
errors = y_test - y_pred
sns.histplot(errors, bins=20, kde=True)
plt.xlabel('Prediction Error (Actual - Predicted)')
plt.ylabel('Frequency')
plt.title('Histogram of Prediction Errors')
plt.grid(True)
plt.show()

##METHOD TWO: GRAPH CONVOLUTIONS

In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.utils import from_networkx
from torch_geometric.data import Data, DataLoader

import networkx as nx
import random

# ---- Generate synthetic data ---- #
def generate_random_graph_data(num_graphs=100):
    data_list = []

    for _ in range(num_graphs):
        G = nx.erdos_renyi_graph(n=random.randint(10, 20), p=0.3)

        # Add dummy node features (e.g., degree as feature)
        for node in G.nodes:
            G.nodes[node]['x'] = [G.degree[node]]

        # Convert to PyG Data object
        data = from_networkx(G)
        data.x = data.x.float()

        # Target: random float (e.g., permeability constant)
        data.y = torch.tensor([random.uniform(0.0, 1.0)], dtype=torch.float)
        data_list.append(data)

    return data_list

# ---- Define a simple GCN model for graph regression ---- #
from torch.nn import Linear

class GCNRegressor(torch.nn.Module):
    def __init__(self, hidden_channels):
        super(GCNRegressor, self).__init__()
        self.conv1 = GCNConv(1, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.lin = Linear(hidden_channels, 1)

    def forward(self, x, edge_index, batch):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        x = F.relu(x)

        # Pooling to get graph-level representation
        x = global_mean_pool(x, batch)
        return self.lin(x)

# ---- Training setup ---- #
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

data_list = generate_random_graph_data(200)
train_loader = DataLoader(data_list[:160], batch_size=16, shuffle=True)
test_loader = DataLoader(data_list[160:], batch_size=16)

model = GCNRegressor(hidden_channels=64).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = torch.nn.MSELoss()

# ---- Training loop ---- #
for epoch in range(1, 101):
    model.train()
    total_loss = 0
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch.x, batch.edge_index, batch.batch)
        loss = loss_fn(out, batch.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch:03d}, Loss: {total_loss / len(train_loader):.4f}")

# ---- Evaluation ---- #
model.eval()
preds = []
labels = []

with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        out = model(batch.x, batch.edge_index, batch.batch)
        preds.append(out.cpu())
        labels.append(batch.y.cpu())

preds = torch.cat(preds)
labels = torch.cat(labels)

# Calculate metrics
mse = F.mse_loss(preds, labels).item()
mae = F.l1_loss(preds, labels).item()
r2 = 1 - mse / torch.var(labels, unbiased=False).item()

print(f"Test MAE: {mae:.4f}, MSE: {mse:.4f}, R²: {r2:.4f}")
